# Recipe Gemma: FULL LoRA fine-tune on Google Colab (GPU: T4 / L4 / A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/train_colab.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/train_colab.ipynb)

Runtime: *Runtime > Change runtime type > GPU* (T4 on the free tier; L4 or A100 on paid tiers).

This is the **full run** (`configs/train_full.yaml`): **Gemma 3 1B IT**, **LoRA r=16**, **seq 512**,
bf16 mixed precision, batch 16. **The T4 has no bf16 support**, so on a T4 the notebook writes
`configs/train_colab.yaml` (extends `train_full.yaml`: `float16` / `mixed_float16`, batch 4, seq 512)
and trains with that. L4 and A100 use `train_full.yaml` unchanged.

**Secrets:** add them in the key icon panel (*Secrets*) and enable notebook access:
`KAGGLE_USERNAME`, `KAGGLE_KEY` (Kaggle-hosted Gemma preset; accept the Gemma license on Kaggle
first), `HF_TOKEN` (optional; needed only for the Hub push) and `GEMINI_API_KEY` (optional; needed only for the
Gemini baseline and judge).

In [ ]:
import sys
assert sys.version_info >= (3, 12), (
    f"Python {sys.version.split()[0]} found; this project (jax 0.12) needs Python >= 3.12. "
    "Switch the runtime to the latest environment/image and restart."
)
print(sys.version)

In [ ]:
# (1) Credentials from Colab Secrets -> os.environ (values are never printed).
import os
from google.colab import userdata

for _name in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        _val = userdata.get(_name)
    except Exception:  # SecretNotFoundError / NotebookAccessError
        _val = None
    if _val:
        os.environ[_name] = _val
    print(f"{_name}: {'set' if _val else 'MISSING'}")
_val = None

In [ ]:
# (2) Clone the repo.
%cd /content
!rm -rf Gemma-Fine-Tuning && git clone --depth 1 https://github.com/nikhil3495/Gemma-Fine-Tuning
%cd /content/Gemma-Fine-Tuning

In [ ]:
# (3) Install with CUDA 12 JAX.
!pip install -q -e ".[train,eval]" safetensors sentencepiece
!pip install -q "jax[cuda12]==0.12.1"
!nvidia-smi --query-gpu=name,memory.total --format=csv

**(4) Backend + license.** Accept the Gemma license on Kaggle
(<https://www.kaggle.com/models/google/gemma-3>, *Request access*) with the `KAGGLE_USERNAME`
account. Otherwise the preset download fails with 403. If you restart the runtime, re-run
the cells from (1).

In [ ]:
# (4) Keras backend (inherited by the `!python` subprocesses).
os.environ["KERAS_BACKEND"] = "jax"
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")
!python -c "import jax; print(jax.devices())"

In [ ]:
# Pick the training config: T4 (no bf16) -> configs/train_colab.yaml, otherwise train_full.yaml.
import subprocess

GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", GPU or "none")

COLAB_YAML = '''# Colab T4 variant of the full run (written by notebooks/train_colab.ipynb).
# T4 has no bf16: fp16 weights + mixed_float16 (Keras adds loss scaling), smaller batch.
# If the loss turns NaN (Gemma 3 activations can overflow fp16), set model.dtype: null (fp32 weights).
extends: train_full.yaml
run_name: full_colab_t4

model:
  preset: gemma3_instruct_1b
  dtype: float16

train:
  batch_size: 4
  seq_len: 512
  eval_batch_size: 4
  mixed_precision: mixed_float16
'''
with open("configs/train_colab.yaml", "w", encoding="utf-8") as fh:
    fh.write(COLAB_YAML)

TRAIN_CONFIG = "configs/train_colab.yaml" if "T4" in GPU else "configs/train_full.yaml"
print("TRAIN_CONFIG =", TRAIN_CONFIG)

In [ ]:
# (5) Build the datasets (downloads one corbt/all-recipes parquet shard + the Ahn et al. flavor data).
!python -m recipe_gemma.data.build --config configs/data.yaml

In [ ]:
# (6) Full LoRA fine-tune. Outputs: checkpoints/full/adapter.lora.h5, results/train_full.json,
# results/train_full_log.csv (train_colab.yaml inherits these paths). Keep the tab open; a free
# Colab session can disconnect before 3000 steps (early stopping usually ends sooner).
!python -m recipe_gemma.train --config {TRAIN_CONFIG}

In [ ]:
# (7) Evaluation + report. configs/eval.yaml decides which adapter/checkpoint is scored; make sure
# it points at the full run (checkpoints/full). The Gemini baseline and the LLM judge need GEMINI_API_KEY;
# without it those rows are reported as "not run".
!python -m recipe_gemma.eval --config configs/eval.yaml
!python -m recipe_gemma.eval.report --config configs/eval.yaml

In [ ]:
# (8) Optional: push the LoRA adapter to the Hugging Face Hub (needs HF_TOKEN with write access).
PUSH_TO_HUB = False
if PUSH_TO_HUB and os.environ.get("HF_TOKEN"):
    !python scripts/push_adapter.py --config {TRAIN_CONFIG}
else:
    print("skipping Hub push (set PUSH_TO_HUB = True and provide HF_TOKEN)")

In [ ]:
# (9) Zip results/ (+ the adapter) for download.
import shutil
from pathlib import Path

OUT = Path("/content")
OUT.mkdir(parents=True, exist_ok=True)
print(shutil.make_archive(str(OUT / "recipe_gemma_results"), "zip", root_dir=".", base_dir="results"))
if Path("checkpoints/full").exists():
    print(shutil.make_archive(str(OUT / "recipe_gemma_adapter_full"), "zip", root_dir=".", base_dir="checkpoints/full"))

from google.colab import files
files.download("/content/recipe_gemma_results.zip")